## Detect functional groups in MOFs
In this notebook, we will use mofography to detect sepecifc functional groups in MOFs. We will then extend this to look for functional groups in a database of MOFs such as the CoRE MOF database. 
### Algorithm 

Under the hood, mofography uses a graph-based approach to detect functional groups. The algorithm is called subgraph isomorphism and is a well-known problem in computer science. The algorithm is implemented in the [networkx](https://networkx.github.io/) package. Detection occurs based on element and bond types. The source code is in `analyze.py` in the mofography package.

### Steps 

- Load the MOF into ASE
- Load the function group as SMARTS
- Convert both to networkx graphs, via. RDKit
- Detect the functional group in the MOF using subgraph isomorphism
- Make separate atoms object for just selected atoms and bonds in the MOF
- Visualize just hat to confirm


### Functional groups
In this example we are just looking for COO groups, which are extremely common in MOFs. If you want to look for other functional groups, please use SMARTS notation to define the functional group. And some explanation of SMARTS is provided later. 

## Imports
Import the necessary packages.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys
path_to_mgr= Path('c:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography')
sys.path.append(str(path_to_mgr))
import mofography as mgr
import pickle
from ase.io import read
import networkx as nx 
from tqdm import tqdm
import plotly.graph_objects as go

## Grab a dataset
Either grab it from MOFDB or if you have pre saved file read it in to a pandas dataframe.

To grab from MOFDB use this one as a mofography example for setting ranges: 

In [4]:
# df = mgr.get_data_from_mofdb(params={'vf_min': 0.7, 'vf_max': 0.75,'lcd_min':12,'lcd_max':14, 'database':'CoREMOF 2019'})

TO read from saved file, for e.g. `.feather` file, use this example: 

In [5]:
df  = pd.read_feather('./core_df_gt_02.feather')

Convert them to ASE atoms objects. DO it once and save it as a pickle file for future use. 

In [ ]:
# core_atoms = mgr.atoms_from_mofdb_df(df)
# pickle.dump(core_atoms, open('core_atoms.pkl','wb'))

Or if you have it already, load it up here: 

In [6]:
core_atoms = pickle.load(open('./core_atoms_list.pkl','rb'))

Let's pick one material and test our methodology

In [7]:
mat_atoms = core_atoms[0]

## How to specify functional moeties? 

Functional moieties, also known as functional groups, can be specified using SMARTS (SMiles ARbitrary Target Specification) notation. SMARTS is a language that allows you to describe molecular patterns. It is used extensively in cheminformatics to search for structural patterns within datasets of molecules.

For example, the SMARTS pattern for a carboxylate group is `[C;X3](-[O;X1])-[O;X1]`.

Here's a breakdown of this pattern:

- `[C;X3]`: This matches a carbon atom (`C`) with three connections (`X3`). The semicolon (`;`) is used to combine different conditions, so `C;X3` matches a carbon atom with exactly three bonds.

- `(-[O;X1])`: This matches an oxygen atom (`O`) with one connection (`X1`), i.e., a single bond. The parentheses are used to group atoms and bonds that should be considered as a unit. The `-` inside the parentheses indicates a single bond.

- `-[O;X1]`: This matches another oxygen atom with a single bond. The `-` outside the parentheses indicates that this oxygen atom is bonded to the previously matched atom (the carbon atom).

So, the SMARTS pattern `[C;X3](-[O;X1])-[O;X1]` matches a carbon atom with three bonds, one of which is to an oxygen atom, and another of which is to another oxygen atom. This is the pattern for a carboxylate group.


In [ ]:
mat_atoms = core_atoms[1]
functional_groups = {
    'carboxylate': '[C;X3](-[O;X1])-[O;X1]',
    # Add more functional groups here
}
matches = mgr.find_functional_groups(mat_atoms, functional_groups)


Let's look at the detected atoms for carboxylate group in the material, if any. 

In [ ]:
matches

## Visualize
Now, let's use mofography to visualize and confirm that the correct atoms are detected. 

In [ ]:
# mat_atoms = read(path_to_cif)
mat_atoms = core_atoms[1]
mat_updated =mgr.update_with_covalent_bonds(mat_atoms)
functional_groups = {
    'carboxylate': '[C;X3](-[O;X1])-[O;X1]',
    # Add more functional groups here
}
matches = mgr.find_functional_groups(mat_updated, functional_groups)
matching_indices = np.concatenate([list(m.keys()) for m in matches['carboxylate']])
# Create a list of atoms that match the indices
# matching_atoms = []
# for group, indices_list in matches.items():
#     for indices in indices_list:
#         matching_atoms.extend([mat_updated[i] for i in indices.keys()])

frame_subset_trace = mgr.plot_subset_of_atoms(mat_updated, subset=matching_indices)


In [ ]:
go.Figure(data=[*frame_subset_trace], layout=mgr.white_layout).show()    

##  Bonus: Visualize the whole material if you're curious

In [ ]:
frame_trace = mgr.plot_framework_as_mesh(mat_updated)
fig  = go.Figure(data=frame_trace)
fig.update(layout=mgr.white_layout)
fig.show(renderer='notebook_connected')

## Apply to CoRE MOFs

Now we have to apply the functional group detection to a list of coremof atoms, that we have precompiled as shown in the beginning of the notebook. We wil use a simple try: except in a for loop for this so that in case some structures fail due to large size or some other reason, we can still screen the rest of the structures without fail.

it is going to take a few 3-5 minutes. 


(PS: If you ahve multiple functional groups, please screen for them all at the same time, not one by one.😂)

In [11]:
functional_groups = {
    'carboxylate': '[C;X3](-[O;X1])-[O;X1]',
    # Add more functional groups here
}
matches_list = []
for atoms in tqdm(core_atoms):
    try:
        matches_list.append(mgr.find_functional_groups(atoms, functional_groups))
    except:
        matches_list.append(None)


100%|██████████| 7509/7509 [03:33<00:00, 35.20it/s] 


In [12]:
matches_list

[{'carboxylate': [{289: 0, 0: 1, 147: 2},
   {289: 0, 147: 1, 0: 2},
   {291: 0, 1: 1, 146: 2},
   {291: 0, 146: 1, 1: 2},
   {293: 0, 4: 1, 151: 2},
   {293: 0, 151: 1, 4: 2},
   {295: 0, 150: 1, 5: 2},
   {295: 0, 5: 1, 150: 2},
   {297: 0, 148: 1, 7: 2},
   {297: 0, 7: 1, 148: 2},
   {299: 0, 8: 1, 159: 2},
   {299: 0, 159: 1, 8: 2},
   {301: 0, 9: 1, 158: 2},
   {301: 0, 158: 1, 9: 2},
   {303: 0, 10: 1, 157: 2},
   {303: 0, 157: 1, 10: 2},
   {305: 0, 155: 1, 12: 2},
   {305: 0, 12: 1, 155: 2},
   {307: 0, 154: 1, 13: 2},
   {307: 0, 13: 1, 154: 2},
   {309: 0, 16: 1, 185: 2},
   {309: 0, 185: 1, 16: 2},
   {311: 0, 18: 1, 187: 2},
   {311: 0, 187: 1, 18: 2},
   {313: 0, 20: 1, 189: 2},
   {313: 0, 189: 1, 20: 2},
   {315: 0, 188: 1, 21: 2},
   {315: 0, 21: 1, 188: 2},
   {317: 0, 22: 1, 191: 2},
   {317: 0, 191: 1, 22: 2},
   {319: 0, 24: 1, 181: 2},
   {319: 0, 181: 1, 24: 2},
   {321: 0, 26: 1, 183: 2},
   {321: 0, 183: 1, 26: 2},
   {323: 0, 27: 1, 182: 2},
   {323: 0, 182: 1,

From here I'll leave it to you to do what ever post processing you want. 